# Column Audit — ตรวจทุกคอลัมน์ เรื่อง naming & format consistency

จุดประสงค์: ไล่ดู **ทุกคอลัมน์ในทุกตาราง** ว่า
1. **format สม่ำเสมอไหม** (dtype, ตัวเลขปนหน่วย, ช่วงค่า, ค่าหลายตัวในช่องเดียว)
2. **ชื่อ/ค่าที่ความหมายเดียวกันถูกเขียนต่างกันไหม** (เช่น `SpinCoating` = `Spin coating`,
   `Sigma-Aldrich` = `Sigma Aldrich`, `mA cm-2` = `mAcm-2`)

แล้วจับกลุ่มที่ความหมายตรงกัน **แสดงผลออกมาเป็นตาราง** พร้อมเสนอ *canonical* (ชื่อมาตรฐาน)
สำหรับเอาไปใช้ในขั้น cleaning ต่อไป และ **export** ข้อเสนอ mapping เป็นไฟล์.

> ข้อมูลตั้งต้น: `processed/csv/*.csv` (ตาราง normalized จาก [`RESCHEMA_README.md`](../RESCHEMA_README.md))

## Setup

In [1]:
import os, re, json
from collections import Counter
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".." if os.path.basename(os.getcwd())=="notebooks" else "."))
os.chdir(ROOT)

TABLES = ["devices","layers","vendors","materials","process","properties","stability"]
t = {name: pd.read_csv(f"processed/csv/{name}.csv") for name in TABLES}

# คอลัมน์ที่ "ตั้งใจให้เป็นตัวเลข" (audit แบบ format ตัวเลข ไม่ใช่ synonym)
QUANT_COLS = {"thickness_nm","amount","parameter","value","error","sample_no",
              "temperature_c","humidity_pct","time_hr","retention_pct","t80_hr",
              "initial_pce","final_pce"}
# id/โครงสร้าง ไม่ต้อง audit เนื้อหา
SKIP_COLS = {"device_id","source_file","source_sheet","layer_id","step_order","doi_raw"}

def kind(col):
    if col in QUANT_COLS: return "quant"
    return "text"

# normalizer สำหรับหา "ความหมายเดียวกันแต่เขียนต่าง" (ใช้กับ text เท่านั้น)
def tkey(v):
    s = str(v).strip().lower()
    s = re.sub(r"[\s\-_/():.,]+", "", s)   # ตัด separator/ช่องว่าง/เครื่องหมาย
    s = s.replace("−", "-")                # unicode minus -> ascii (ก่อนถูกตัดด้านบนแล้ว)
    return s

def variant_groups(series, keyfn=tkey):
    # คืนกลุ่มค่าที่ normalize แล้วตรงกัน แต่สะกดต่างกัน (>1 รูปแบบ)
    s = series.dropna().astype(str).map(str.strip)
    s = s[s != ""]
    groups = {}
    for v in s:
        groups.setdefault(keyfn(v), []).append(v)
    rows = []
    for k, vals in groups.items():
        c = Counter(vals)
        if len(c) > 1:
            rows.append({"canonical_key": k, "n_variants": len(c),
                         "variants": "  |  ".join(f"{x} ×{c[x]}" for x in sorted(c)),
                         "total_rows": sum(c.values())})
    out = pd.DataFrame(rows)
    return out.sort_values(["n_variants","total_rows"], ascending=False).reset_index(drop=True) if len(out) else out

print("โหลด", len(t), "ตาราง:", {k: v.shape for k, v in t.items()})

โหลด 7 ตาราง: {'devices': (44, 4), 'layers': (352, 5), 'vendors': (644, 4), 'materials': (543, 7), 'process': (702, 9), 'properties': (563, 6), 'stability': (62, 10)}


## 1. Master column inventory — ภาพรวมทุกคอลัมน์

ตารางเดียวเห็นทุกคอลัมน์: ชนิด, %missing, จำนวนค่าไม่ซ้ำ, จำนวนกลุ่มที่สะกดต่าง (text),
และอัตราที่ parse เป็นตัวเลขได้ (quant).

In [2]:
def numeric_parse_rate(series):
    x = series.dropna().astype(str).str.strip()
    if len(x) == 0: return np.nan
    pure = x.str.fullmatch(r"-?\d+\.?\d*")
    return round(pure.mean(), 3)

rows = []
for name in TABLES:
    df = t[name]
    for col in df.columns:
        if col in SKIP_COLS: continue
        s = df[col]
        nn = s.notna().sum()
        info = {"table": name, "column": col, "kind": kind(col),
                "n_nonnull": int(nn), "pct_missing": round(s.isna().mean()*100, 1),
                "n_unique": int(s.dropna().astype(str).str.strip().nunique())}
        if kind(col) == "text":
            vg = variant_groups(s)
            info["variant_groups"] = 0 if len(vg)==0 else int(len(vg))
            info["numeric_parse_rate"] = np.nan
        else:
            info["variant_groups"] = np.nan
            info["numeric_parse_rate"] = numeric_parse_rate(s)
        rows.append(info)

inventory = pd.DataFrame(rows)
inventory

,table,column,kind,n_nonnull,pct_missing,n_unique,variant_groups,numeric_parse_rate
0,devices,sample_no,quant,44,0.0,5,NaN,1.000
1,layers,layer_type,text,349,0.9,12,2.0,NaN
2,layers,material,text,315,10.5,65,0.0,NaN
3,layers,thickness_nm,quant,105,70.2,25,NaN,0.886
4,vendors,material,text,644,0.0,79,0.0,NaN
5,vendors,purity,text,346,46.3,22,0.0,NaN
6,vendors,vendor,text,562,12.7,59,2.0,NaN
7,materials,material,text,511,5.9,61,0.0,NaN
8,materials,type,text,375,30.9,22,1.0,NaN
9,materials,solvent,text,337,37.9,37,6.0,NaN


## 2. ⭐ Text columns — ค่าที่ "ความหมายเดียวกันแต่เขียนต่าง"

หัวใจของคำขอ: รวมทุก text column แล้วแสดงกลุ่มที่ normalize (ตัดช่องว่าง/ตัวพิมพ์/เครื่องหมาย)
แล้วตรงกัน แต่สะกดไม่เหมือนกัน — เรียงจากกลุ่มที่มีหลายรูปแบบสุด.

In [3]:
allvg = []
for name in TABLES:
    df = t[name]
    for col in df.columns:
        if col in SKIP_COLS or kind(col) != "text": continue
        vg = variant_groups(df[col])
        if len(vg):
            vg.insert(0, "table", name); vg.insert(1, "column", col)
            allvg.append(vg)
allvg = pd.concat(allvg, ignore_index=True).sort_values(
    ["n_variants","total_rows"], ascending=False).reset_index(drop=True)
print(f"พบ {len(allvg)} กลุ่ม ที่ความหมายเดียวกันแต่เขียนต่างกัน\n")
allvg[["table","column","n_variants","total_rows","variants"]]

พบ 35 กลุ่ม ที่ความหมายเดียวกันแต่เขียนต่างกัน



,table,column,n_variants,total_rows,variants
0,materials,solvent,4,114,DMF/DMSO (4/1 V/V) ×51 | DMF/DMSO (4:1 V/V) ×43 | DMF/DMSO (4:1 v/v) ×8 | DMF:DMSO (4:1 v/v) ×12
1,properties,unit,4,69,mA cm-2 ×52 | mA/cm-2 ×2 | mA/cm2 ×3 | mAcm-2 ×12
2,materials,solvent,4,6,DMF /DMSO (3.08/1 V/V) ×1 | DMF/DMSO (3.08/1 V/V) ×1 | DMF/DMSO (3.08/1 v/v) ×2 | DMFDMSO (3.08/1 v/v) ×2
3,process,process_type,3,82,Thermal Evaporation ×12 | Thermal evaporation ×6 | ThermalEvaporation ×64
4,process,process_type,2,213,Spin coating ×9 | SpinCoating ×204
5,process,unit,2,157,"C, sec ×153 | C,sec ×4"
6,vendors,vendor,2,100,Sigma Aldrich ×12 | Sigma-Aldrich ×88
7,layers,layer_type,2,43,HTL Treatment ×1 | HTL treatment ×42
8,process,process_type,2,42,AntiSolvent ×35 | Antisolvent ×7
9,layers,layer_type,2,40,PSK Treatment ×1 | PSK treatment ×39


## 3. Quantitative columns — ตรวจ format ของตัวเลข

คอลัมน์ที่ควรเป็นตัวเลขแต่เก็บเป็น string: ดูว่ากี่ % เป็นเลขล้วน, กี่ % ปนตัวอักษร/หน่วย,
เป็นช่วง (`a - b`), หรือหลายค่าในช่องเดียว (`80,12`).

In [4]:
def quant_profile(series):
    x = series.dropna().astype(str).map(str.strip)
    x = x[x != ""]
    n = len(x)
    if n == 0: return None
    pure   = x.str.fullmatch(r"-?\d+\.?\d*")
    has_a  = x.str.contains(r"[A-Za-zΩµμÅ°%]")
    rng    = x.str.contains(r"\d\s*[-–~]\s*\d")
    multi  = x.str.contains(r",|/| x ", regex=True)
    return {"n": n,
            "pure_number_%": round(pure.mean()*100,1),
            "has_letters/unit_%": round(has_a.mean()*100,1),
            "range_%": round(rng.mean()*100,1),
            "multivalue_%": round(multi.mean()*100,1),
            "examples_dirty": " | ".join(x[~pure].unique()[:5])}

rows = []
for name in TABLES:
    df = t[name]
    for col in df.columns:
        if col in SKIP_COLS or kind(col) != "quant": continue
        p = quant_profile(df[col])
        if p: rows.append({"table": name, "column": col, **p})
quant_report = pd.DataFrame(rows)
quant_report

,table,column,n,pure_number_%,has_letters/unit_%,range_%,multivalue_%,examples_dirty
0,devices,sample_no,44,100.0,0.0,0.0,0.0,
1,layers,thickness_nm,105,88.6,5.7,3.8,1.9,"400-450 | 4, 100 | 400 Å | 90 Å | 1200 Å"
2,materials,amount,323,100.0,0.0,0.0,0.0,
3,process,parameter,595,26.4,4.2,1.7,69.4,"3000, 30 | 270, 10800 | 60, 3600 | 1000,500,10 | 4000,2000,35"
4,properties,value,544,96.9,3.1,0.0,0.0,3.3e-08 | 3.06e-08 | 2.84e-08 | 8.1e+18 | 1.51e+18
5,properties,error,74,100.0,0.0,0.0,0.0,
6,stability,temperature_c,48,87.5,12.5,0.0,0.0,RT
7,stability,humidity_pct,26,53.8,0.0,26.9,0.0,50 - 60 | 50–60 | 45–55 | 30-50 | 60 ± 5
8,stability,time_hr,59,100.0,0.0,0.0,0.0,
9,stability,retention_pct,50,98.0,0.0,0.0,0.0,< 55


## 4. Deep-dives + canonical mapping ที่เสนอ

ต่อไปนี้ไล่ทีละคอลัมน์สำคัญ พร้อมเสนอ **ชื่อมาตรฐาน (canonical)** และแสดง raw→canonical.
`CANON` แต่ละชุดคือ dict ที่เอาไปใช้ในขั้น cleaning ได้เลย.

### 4.1 `process.process_type` — ขั้นตอนการผลิต (จุดเด่นของโปรเจกต์)

In [5]:
PROCESS_CANON = {
 "spincoating":"spin_coating", "spin coating":"spin_coating",
 "annealing":"annealing", "postannealing":"post_annealing",
 "thermalevaporation":"thermal_evaporation", "thermal evaporation":"thermal_evaporation",
 "electronbeamevaporation":"ebeam_evaporation",
 "antisolvent":"antisolvent",
 "wash":"wash", "ethanolwash":"wash", "dry":"dry",
 "plasmatreat":"plasma_treatment", "plasma treat":"plasma_treatment",
 "uv/ozone treat":"uv_ozone_treatment", "uv treat":"uv_treatment", "treatment":"treatment",
 "stir":"stir", "magnetic stir":"stir",
 "mix":"mixing", "mixing":"mixing",
 "sonication":"sonication", "dip-coating":"dip_coating", "dissolve":"dissolve",
 "ph adjustment":"ph_adjustment",
 "sputtering":"sputtering", "rfsputtering":"sputtering", "magnetronsputtering":"sputtering",
 "presputtering":"pre_sputtering", "pre-sputtering":"pre_sputtering",
 "ald":"ald",
}
def map_report(series, canon):
    vc = series.dropna().astype(str).str.strip().value_counts()
    df = vc.rename_axis("raw").reset_index(name="count")
    df["canonical"] = df["raw"].str.lower().map(canon)
    return df

pt = map_report(t["process"]["process_type"], PROCESS_CANON)
unmapped = pt[pt["canonical"].isna()]
print(f"raw {pt['raw'].nunique()} แบบ -> canonical {pt['canonical'].nunique()} แบบ | "
      f"map ไม่เจอ {len(unmapped)} แบบ ({unmapped['count'].sum()} rows)")
pt

raw 32 แบบ -> canonical 21 แบบ | map ไม่เจอ 0 แบบ (0 rows)


,raw,count,canonical
0,SpinCoating,204,spin_coating
1,Annealing,161,annealing
2,ThermalEvaporation,64,thermal_evaporation
3,Wash,52,wash
4,AntiSolvent,35,antisolvent
5,Dry,32,dry
6,Plasma Treat,17,plasma_treatment
7,UV/Ozone Treat,14,uv_ozone_treatment
8,Thermal Evaporation,12,thermal_evaporation
9,Spin coating,9,spin_coating


### 4.2 `layers.layer_type` และ `materials.type`

In [6]:
LAYER_CANON = {  # รวมความต่างเรื่องตัวพิมพ์
 "htl treatment":"HTL treatment", "psk treatment":"PSK treatment",
 "etl treatment":"ETL treatment",
}
lt = t["layers"]["layer_type"].dropna().str.strip()
lt_df = lt.value_counts().rename_axis("raw").reset_index(name="count")
lt_df["canonical"] = lt_df["raw"].str.lower().map(LAYER_CANON).fillna(lt_df["raw"])
print("layers.layer_type:")
display(lt_df)

TYPE_CANON = {"anti-solvent":"antisolvent", "antisolvent":"antisolvent",
              "nanoparticle":"nanoparticles", "nanoparticles":"nanoparticles"}
ty = t["materials"]["type"].dropna().str.strip()
ty_df = ty.value_counts().rename_axis("raw").reset_index(name="count")
ty_df["canonical"] = ty_df["raw"].str.lower().map(TYPE_CANON).fillna(ty_df["raw"])
print("materials.type:")
ty_df

layers.layer_type:


,raw,count,canonical
0,Anode,44,Anode
1,HTL,44,HTL
2,PSK,44,PSK
3,ETL,44,ETL
4,Cathode,44,Cathode
5,HTL treatment,42,HTL treatment
6,ETL treatment,42,ETL treatment
7,PSK treatment,39,PSK treatment
8,ETL treatment (1),2,ETL treatment (1)
9,ETL treatment (2),2,ETL treatment (2)


materials.type:


,raw,count,canonical
0,Precursor,155,Precursor
1,Solution,126,Solution
2,Additive,29,Additive
3,Substrate,19,Substrate
4,Nanoparticles,9,nanoparticles
5,SAM,5,SAM
6,Interlayer,4,Interlayer
7,Antisolvent,4,antisolvent
8,Surface modifier,4,Surface modifier
9,PSK surface treatment,3,PSK surface treatment


### 4.3 ชื่อวัสดุ (`material`) — จับกลุ่ม chemistry family

ชื่อวัสดุปรากฏใน 3 ตาราง (`layers`, `materials`, `vendors`). ตรวจว่า variant เดียวกันไหม
และจับตระกูล เช่น `NiOx / NiOx NPs / NiOx Nanoparticles` = NiOx.

In [7]:
def material_family(x):
    s = str(x).lower()
    if "nio" in s: return "NiOx"
    if s.startswith("c60") or s=="c60": return "C60"
    if "bcp" in s: return "BCP"
    if "pbi2" in s: return "PbI2"
    if "pedot" in s: return "PEDOT:PSS"
    if "spiro" in s: return "Spiro-OMeTAD"
    if any(k in s for k in ["pacz","2pacz","me-2pacz","mecz"]): return "SAM (xPACz)"
    return None

mats = pd.concat([t["layers"][["material"]].assign(src="layers"),
                  t["materials"][["material"]].assign(src="materials"),
                  t["vendors"][["material"]].assign(src="vendors")], ignore_index=True)
mats = mats.dropna()
mats["material"] = mats["material"].astype(str).str.strip()
mats["family"] = mats["material"].map(material_family)
fam = (mats.dropna(subset=["family"]).groupby("family")["material"]
          .agg(lambda s: "  |  ".join(f"{v} ×{c}" for v,c in Counter(s).most_common()))
          .reset_index(name="spellings_found"))
fam["n_spellings"] = fam["spellings_found"].str.count(r"\|") + 1
print("ตระกูลวัสดุที่มีหลายชื่อ:")
fam.sort_values("n_spellings", ascending=False)

ตระกูลวัสดุที่มีหลายชื่อ:


,family,spellings_found,n_spellings
2,NiOx,NiOx ×51 | NiOx NPs ×35 | NiO target ×6 | NiOx Nanoparticles ×5 | NiOx NPs ink ×2 | Cu:NiO ×1 | NiOx:TTT...,9
4,SAM (xPACz),Me-4PACz ×10 | 2PACz ×4 | Me-2PACz ×4 | Me-2PACz (SAM) ×2 | Me-4PACz + PFPTFB ×1 | 2PACz SAM ×1 | Me-4PA...,9
0,BCP,"BCP ×111 | PbI2, BCP ×5 | BCP (bathocuproine) ×2 | BCP + SnO2 ×1",4
1,C60,C60 ×52,1
3,PbI2,PbI2 ×78,1


### 4.4 `materials.solvent` — สูตรตัวทำละลายเขียนต่างกัน

In [8]:
variant_groups(t["materials"]["solvent"])[["canonical_key","n_variants","variants"]]

,canonical_key,n_variants,variants
0,dmfdmso41vv,4,DMF/DMSO (4/1 V/V) ×51 | DMF/DMSO (4:1 V/V) ×43 | DMF/DMSO (4:1 v/v) ×8 | DMF:DMSO (4:1 v/v) ×12
1,dmfdmso3081vv,4,DMF /DMSO (3.08/1 V/V) ×1 | DMF/DMSO (3.08/1 V/V) ×1 | DMF/DMSO (3.08/1 v/v) ×2 | DMFDMSO (3.08/1 v/v) ×2
2,dmfdmso41,2,DMF/DMSO (4:1) ×8 | DMF:DMSO (4:1) ×10
3,ethanol,2,Ethanol ×1 | ethanol ×7
4,deionizedwater,2,De-ionized water ×3 | Deionized water ×4
5,ethylalcohol,2,Ethyl Alcohol ×2 | ethyl alcohol ×2


### 4.5 หน่วย (`unit`) — 3 คอลัมน์ (materials / process / properties)

หน่วยยุ่งที่สุด. แสดง vocabulary จริง + เสนอ canonical.
⚠️ ระวัง: `mg/mL` ≠ `mg/L`, `Ω cm²` ≠ `Ω/cm²` — ต่างความหมาย ห้าม merge มั่ว.

In [9]:
PROP_UNIT_CANON = {
 "mA cm-2":"mA/cm²","mAcm-2":"mA/cm²","mA cm−2":"mA/cm²","mA/cm2":"mA/cm²","mA/cm-2":"mA/cm²",
 "Ω":"ohm","Ohm":"ohm","Ω cm²":"ohm·cm²","Ω cm2":"ohm·cm²","Ohm/cm2":"ohm/cm²","Ω/sq":"ohm/sq",
 "deg":"degree","degrees":"degree","S cm-1":"S/cm","cm2 V-1 s-1":"cm²/(V·s)",
 "cm2/Vs":"cm²/(V·s)","m2 V-1 s-1":"m²/(V·s)",
}
pu = t["properties"]["unit"].dropna().str.strip().value_counts().rename_axis("raw").reset_index(name="count")
pu["canonical"] = pu["raw"].map(PROP_UNIT_CANON).fillna(pu["raw"])
print("properties.unit (เสนอ canonical เฉพาะที่ชัด):")
display(pu)
print("\nmaterials.unit vocabulary:")
display(t["materials"]["unit"].dropna().str.strip().value_counts().rename_axis("raw").reset_index(name="count"))
print("process.unit vocabulary (เป็น composite เช่น 'RPM, sec' -> ต้องแตกทีหลัง):")
t["process"]["unit"].dropna().str.strip().value_counts().rename_axis("raw").reset_index(name="count")

properties.unit (เสนอ canonical เฉพาะที่ชัด):


,raw,count,canonical
0,%,161,%
1,V,96,V
2,mA cm-2,52,mA/cm²
3,eV,50,eV
4,nm,45,nm
5,cm-3,30,cm-3
6,Ω,14,ohm
7,deg,14,degree
8,ns,12,ns
9,mAcm-2,12,mA/cm²



materials.unit vocabulary:


,raw,count
0,mg,97
1,mg mL-1,88
2,M,39
3,mol%,15
4,mg/mL,15
5,µL,9
6,g / 2mL,8
7,mg/ml,7
8,mol,6
9,mmol,6


process.unit vocabulary (เป็น composite เช่น 'RPM, sec' -> ต้องแตกทีหลัง):


,raw,count
0,"RPM, sec",191
1,"C, sec",153
2,sec,100
3,Torr,17
4,C,16
5,"μL, sec",13
6,"Å/s, Torr",12
7,M,8
8,"C, hr",8
9,Pa,7


### 4.6 `vendors.vendor` — ชื่อผู้ผลิต

In [10]:
variant_groups(t["vendors"]["vendor"])[["canonical_key","n_variants","variants"]]

,canonical_key,n_variants,variants
0,sigmaaldrich,2,Sigma Aldrich ×12 | Sigma-Aldrich ×88
1,xianpolymerlighttechnologycorp,2,Xi-an Polymer Light Technology Corp. ×16 | Xian Polymer Light Technology Corp ×5


### 4.7 `properties.property` — แตกเป็น base metric + เงื่อนไขการวัด

99 ชื่อ property ส่วนใหญ่คือ metric เดียวกันแต่ต่าง scan/สถิติ
(เช่น `Champion PCE`, `Average PCE`, `PCE (RS)`, `PCE (FS)` = PCE คนละเงื่อนไข).
แตกออกเป็น `base` (Voc/Jsc/FF/PCE/…) + `condition` (champion/average/RS/FS/…).

In [11]:
def base_metric(name):
    n = str(name).lower()
    for key, base in [("pce","PCE"),("voc","Voc"),("jsc","Jsc"),
                      ("ff","FF"),("band","bandgap"),("vbm","VBM"),
                      ("work function","work_function"),("trap","trap_density"),
                      ("grain","grain_size"),("contact angle","contact_angle"),
                      ("mobility","mobility"),("rms","roughness_rms"),
                      ("lifetime","carrier_lifetime")]:
        if re.search(rf"\b{re.escape(key)}\b", n) or key in n:
            return base
    return "other"

def condition(name):
    n = str(name).lower()
    tags = []
    if "champion" in n or "best" in n: tags.append("champion")
    if "average" in n or "avg" in n or "mean" in n: tags.append("average")
    if "certified" in n or "verified" in n: tags.append("certified")
    if re.search(r"\(?\brs\b\)?|reverse", n): tags.append("RS")
    if re.search(r"\(?\bfs\b\)?|forward", n): tags.append("FS")
    if "eqe" in n: tags.append("EQE")
    return "+".join(tags) if tags else "—"

prop = t["properties"]["property"].dropna().str.strip()
pdf = prop.value_counts().rename_axis("raw").reset_index(name="count")
pdf["base"] = pdf["raw"].map(base_metric)
pdf["condition"] = pdf["raw"].map(condition)
print("สรุปตาม base metric (รวมทุกเงื่อนไข):")
display(pdf.groupby("base")["count"].sum().sort_values(ascending=False).reset_index())
print("\nตัวอย่าง JV metrics ที่ base เดียวกันแต่หลายชื่อ:")
pdf[pdf["base"].isin(["PCE","Voc","Jsc","FF"])].sort_values(["base","count"], ascending=[True,False])

สรุปตาม base metric (รวมทุกเงื่อนไข):


,base,count
0,other,151
1,Jsc,79
2,PCE,74
3,FF,67
4,Voc,67
5,grain_size,27
6,trap_density,26
7,contact_angle,18
8,VBM,15
9,roughness_rms,13



ตัวอย่าง JV metrics ที่ base เดียวกันแต่หลายชื่อ:


,raw,count,base,condition
3,Champion FF,20,FF,champion
9,Average FF,13,FF,average
17,FF (RS),9,FF,RS
30,FF,7,FF,—
33,FF (FS),7,FF,FS
42,Champion FF (FS),4,FF,champion+FS
47,Champion FF (RS),4,FF,champion+RS
78,Average FF (RS),2,FF,average+RS
93,Avg FF,1,FF,average
5,Champion Jsc,20,Jsc,champion


### 4.8 `stability.environment` — จับกลุ่มบรรยากาศ/โปรโตคอล

ปนกันหลายมิติ: บรรยากาศ (N2/Air), โปรโตคอล ISOS, โหมดวัด (MPP/MPPT). เสนอ `atmosphere`.

In [12]:
def atmosphere(x):
    s = str(x).lower()
    if "n2" in s or "nitrogen" in s: return "N2/inert"
    if "ambient" in s or "air" in s or "humidity" in s: return "ambient/air"
    if "thermal" in s: return "thermal"
    if "mpp" in s or "operational" in s: return "operational(MPP)"
    if "isos" in s: return "ISOS (unspecified atm.)"
    return "other"
env = t["stability"]["environment"].dropna().str.strip()
edf = env.value_counts().rename_axis("raw").reset_index(name="count")
edf["atmosphere"] = edf["raw"].map(atmosphere)
edf.sort_values(["atmosphere","count"], ascending=[True,False])

,raw,count,atmosphere
2,ISOS-L2,4,ISOS (unspecified atm.)
18,ISOS-L-3,1,ISOS (unspecified atm.)
0,N2,11,N2/inert
3,N2/ISOS-L-2I,4,N2/inert
4,Nitrogen (N2),3,N2/inert
6,N2/ISOS-D-2,3,N2/inert
11,N2 atmosphere,2,N2/inert
13,"N2 glovebox, AM1.5G 1-sun illumination",2,N2/inert
17,N2 storage,1,N2/inert
22,N2/ISOS-L-2I (with top passivation),1,N2/inert


## 5. Cross-table consistency — ชื่อวัสดุข้ามตารางตรงกันไหม

วัสดุตัวเดียวกันควรสะกดเหมือนกันใน `layers`, `materials`, `vendors`.
หา normalized-key ที่มีหลายสะกด "ข้ามตาราง".

In [13]:
frames = []
for name in ["layers","materials","vendors"]:
    s = t[name]["material"].dropna().astype(str).str.strip()
    frames.append(pd.DataFrame({"table": name, "material": s, "key": s.map(tkey)}))
allm = pd.concat(frames, ignore_index=True)
rows = []
for k, g in allm.groupby("key"):
    spell = sorted(g["material"].unique())
    if len(spell) > 1:
        rows.append({"key": k, "n_spellings": len(spell),
                     "spellings": "  |  ".join(spell),
                     "tables": ", ".join(sorted(g["table"].unique()))})
cross = pd.DataFrame(rows, columns=["key","n_spellings","spellings","tables"])
if len(cross):
    cross = cross.sort_values("n_spellings", ascending=False).reset_index(drop=True)
print(f"{len(cross)} วัสดุที่สะกดไม่ตรงกัน (normalize แล้ว key เดียวกันแต่หลายสะกด)")
cross if len(cross) else "✅ ชื่อวัสดุสะกดสม่ำเสมอ (ภายใต้ normalize) ทั้งใน layers/materials/vendors"

0 วัสดุที่สะกดไม่ตรงกัน (normalize แล้ว key เดียวกันแต่หลายสะกด)


'✅ ชื่อวัสดุสะกดสม่ำเสมอ (ภายใต้ normalize) ทั้งใน layers/materials/vendors'

## 6. Export ข้อเสนอ mapping

เขียนออกเป็น `processed/audit/` เพื่อใช้ต่อในขั้น cleaning:
- `variant_groups_text.csv` — ทุกกลุ่มที่สะกดต่าง (§2)
- `quant_format_report.csv` — ปัญหา format ตัวเลข (§3)
- `canonical_maps.json` — dict มาตรฐานที่เสนอ (process_type, units, ฯลฯ)

In [14]:
os.makedirs("processed/audit", exist_ok=True)
allvg.to_csv("processed/audit/variant_groups_text.csv", index=False)
quant_report.to_csv("processed/audit/quant_format_report.csv", index=False)
canonical_maps = {
    "process_type": PROCESS_CANON,
    "layer_type": LAYER_CANON,
    "materials_type": TYPE_CANON,
    "properties_unit": PROP_UNIT_CANON,
}
with open("processed/audit/canonical_maps.json", "w") as f:
    json.dump(canonical_maps, f, indent=2, ensure_ascii=False)
print("wrote -> processed/audit/{variant_groups_text.csv, quant_format_report.csv, canonical_maps.json}")
print("text variant groups:", len(allvg), "| quant cols profiled:", len(quant_report))

wrote -> processed/audit/{variant_groups_text.csv, quant_format_report.csv, canonical_maps.json}
text variant groups: 35 | quant cols profiled: 12


## สรุป (Key findings)

**Format ที่ยังไม่สม่ำเสมอ (ต้อง clean)**
- **ตัวพิมพ์/ช่องว่าง/เครื่องหมาย**: `SpinCoating`=`Spin coating`, `Sigma-Aldrich`=`Sigma Aldrich`,
  `HTL treatment`=`HTL Treatment`, `Ambient`=`ambient`, `mg/mL`=`mg/ml`
- **หน่วยเขียนหลายแบบ**: `mA cm-2 / mAcm-2 / mA cm−2 / mA/cm2` (มี unicode minus `−` ปนด้วย)
- **ตัวเลขปนหน่วย/หลายค่า**: `process.parameter` และ `process.unit` เป็น composite
  (`"80,12"` คู่กับ `"C, hr"`) — ต้องแตกเป็นหลายคอลัมน์
- **ค่าน่าสงสัย**: `properties.value` มี `1.02` กับ `10.2`, `-4.77` กับ `4.77` (ตำแหน่งจุด/เครื่องหมาย)

**ความหมายเดียวกัน–คนละชื่อ (จับกลุ่มแล้วใน §2, §4)**
- วัสดุ: `NiOx / NiOx NPs / NiOx Nanoparticles` ฯลฯ (ดู §4.3, §5)
- property: base เดียวกันหลายเงื่อนไข `Champion/Average/RS/FS PCE` (ดู §4.7)
- solvent, vendor, process_type, environment — มี canonical เสนอไว้แล้ว

**ขั้นต่อไป**: ใช้ `processed/audit/canonical_maps.json` + variant groups เขียน **cleaning layer**
(normalize ชื่อ, แตกหน่วย, แก้ค่าน่าสงสัย) ก่อนทำ feature engineering / โมเดล